# Cross-Lingual Identifiable Victim Effect (IVE) Benchmark
## A Cross-Lingual Study of LLM Moral Allocation Bias

This notebook provides a complete, self-contained replication environment for evaluating the Identifiable Victim Effect across **English, Hindi, and Spanish** on the preregistered 9-model judge panel.

**Hardware Target**: NVIDIA T4 (16GB) or P100 (16GB) on Kaggle (Accelerator: GPU T4 x2 or P100).

In [ ]:
# Step 1: Environment Setup & Dependencies
!pip install -q torch transformers accelerate bitsandbytes pydantic scipy statsmodels pandas numpy matplotlib seaborn pyyaml huggingface_hub

In [ ]:
# Step 2: Hugging Face Authentication (Required for gated models like Llama 3.1 & Gemma 3)
import os
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    if hf_token:
        login(token=hf_token)
        os.environ["HF_TOKEN"] = hf_token
        print("Successfully authenticated with Hugging Face via Kaggle Secrets.")
    else:
        print("No HF_TOKEN found in Kaggle Secrets.")
except Exception as e:
    print("Hugging Face login note:", e)

In [ ]:
# Step 3: Run Preflight Hardware & Environment Audit
!python experiments/kaggle_preflight.py

In [ ]:
# Step 4: Mandatory Smoke Test (1 model × 1 scenario × 1 lang × 2 conditions = 2 judgments)
# Confirms GPU model inference and JSON parser work end-to-end.
!python experiments/run_pilot.py --smoke-test

In [ ]:
# Step 5: Run Language Comprehension Control Battery
!python experiments/run_language_control.py

In [ ]:
# Step 6: Execute Scientific Pilot (10 scenarios × 3 langs × 2 conditions × 9 models = 540 judgments)
# For full 20-scenario benchmark (1,080 judgments), use: !python experiments/run_pilot.py --full-benchmark
!python experiments/run_pilot.py --scientific-pilot

In [ ]:
# Step 7: End-to-End Statistical Analysis, Bootstrapping, and Figure Generation
!python reproduce.py

In [ ]:
# Step 8: Display Publication Figures
from IPython.display import Image, display
from pathlib import Path

fig_dir = Path("results/figures")
for p in sorted(list(fig_dir.glob("*.png"))):
    print(f"\nDisplaying: {p.name}")
    display(Image(filename=str(p)))